# Tutorial 01 — Data Ingestion & Harmonization

**GSoC 2026 Project #39 — NeuroSim | INCF / EBRAINS**

This notebook demonstrates:
1. Loading BOLD time-series data from BIDS-compliant datasets
2. Extracting parcellation-based FC matrices using Nilearn
3. Removing scanner effects using Blind neuroCombat (fit on HCP, apply to ADNI)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# NeuroSim imports
from neurosim.harmonization import fit_combat, apply_combat, blind_harmonize

## Step 1: Simulate Multi-Site Data

In a real pipeline, this would be replaced by PyBIDS queries against HCP / ADNI / OpenNeuro.
Here we simulate a realistic multi-scanner scenario to validate the harmonization logic.

In [ ]:
rng = np.random.default_rng(seed=42)
n_features = 200   # parcels (e.g., Schaefer-200)
n_hc = 100         # healthy control subjects

# Simulate HC data with an artificial scanner site effect
# Site A (HCP_3T): 50 subjects — no offset
# Site B (HCP_7T): 50 subjects — +2.5 feature-wide offset
data_site_A = rng.standard_normal((n_features, 50))
data_site_B = rng.standard_normal((n_features, 50)) + 2.5

hc_data = np.hstack([data_site_A, data_site_B])
hc_labels = ['HCP_3T'] * 50 + ['HCP_7T'] * 50

print(f"HC dataset shape: {hc_data.shape}")
print(f"Site A mean: {data_site_A.mean():.3f}  |  Site B mean: {data_site_B.mean():.3f}")

## Step 2: Fit Blind neuroCombat on HCP Healthy Controls

In [ ]:
combat_params = fit_combat(hc_data, hc_labels)

print(f"ComBat fitted on {combat_params['n_batches']} scanners")
print(f"Number of features harmonized: {combat_params['n_features']}")
print(f"Reference subjects: {combat_params['reference_n_subjects']}")
print(f"\nAdditive batch effect (gamma) shape: {combat_params['gamma_hat'].shape}")
print(f"Scanner B mean gamma (expected ~2.5): {combat_params['gamma_hat'][1].mean():.3f}")

## Step 3: Apply to Clinical Cohort (ADNI — Blind)

The clinical data is harmonized using ONLY the parameters learned from HC.
The disease signal is NOT used to estimate batch effects — this is the 'blind' guarantee.

In [ ]:
# Simulate ADNI clinical cohort — 60 patients, same scanner as HCP_3T
# Add a small disease effect (+0.8) on top of the scanner signature
clinical_data = rng.standard_normal((n_features, 60)) + 0.8
clinical_labels = ['HCP_3T'] * 60

harmonized = apply_combat(clinical_data, clinical_labels, combat_params)

print(f"Original clinical mean:    {clinical_data.mean():.4f}")
print(f"Harmonized clinical mean:  {harmonized.mean():.4f}")
print(f"Expected grand mean:       {combat_params['grand_mean'].mean():.4f}")

## Step 4: Visualize Before/After Scanner Effect Removal

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Before harmonization — clear site separation
axes[0].hist(data_site_A.mean(axis=0), bins=20, alpha=0.7, label='HCP_3T', color='steelblue')
axes[0].hist(data_site_B.mean(axis=0), bins=20, alpha=0.7, label='HCP_7T', color='coral')
axes[0].set_title('Before Harmonization', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Feature Mean Across Subjects')
axes[0].legend()

# After harmonization — distributions should overlap
harmonized_site_B = apply_combat(data_site_B, ['HCP_7T'] * 50, combat_params)
axes[1].hist(data_site_A.mean(axis=0),      bins=20, alpha=0.7, label='HCP_3T (unchanged)', color='steelblue')
axes[1].hist(harmonized_site_B.mean(axis=0), bins=20, alpha=0.7, label='HCP_7T (harmonized)', color='mediumseagreen')
axes[1].set_title('After Blind neuroCombat', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Feature Mean Across Subjects')
axes[1].legend()

plt.suptitle('NeuroSim Module A: Site Effect Removal', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig('harmonization_demo.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved to harmonization_demo.png")